In [13]:
from pathlib import Path

import duckdb

con = duckdb.connect()
data = Path('data')
train = con.read_csv(str(data / 'train.csv' / 'train.csv'), header=True)
members = con.read_csv(str(data / 'members_v3.csv' / 'members_v3.csv'), header=True)
display(train)


msnoVARCHAR,is_churnBIGINT
waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1
QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1
fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1
mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1
XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1
GBy8qSz16X5iYWD+3CMxv/Hm6OPSrXBYtmbnlRtknW0=,1
lYLh7TdkWpIoQs3i3o6mIjLH8/IEgMWP9r7OpsLX0Vo=,1
T0FF6lumjKcqEO0O+tUH2ytc+Kb9EkeaLzcVUiTr1aE=,1
Nb1ZGEmagQeba5E+nQj8VlQoWl+8SFmLZu+Y8ytIamw=,1
MkuWz0Nq6/Oq5fKqRddWL7oh2SLUSRe3/g+XmAWqW1Q=,1


In [14]:
display(members)

msnoVARCHAR,cityBIGINT,bdBIGINT,genderVARCHAR,registered_viaBIGINT,registration_init_timeBIGINT
Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NULL,11,20110911
+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NULL,7,20110914
cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NULL,11,20110915
9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NULL,11,20110915
WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915
yLkV2gbZ4GLFwqTOXLVHz0VGrMYcgBGgKZ3kj9RiYu8=,4,30,male,9,20110916
jNCGK78YkTyId3H3wFavcBLDmz7pfqlvCfUKf4G1Lw4=,1,0,NULL,7,20110916
WH5Jq4mgtfUFXh2yz+HrcTXKS4Oess4k4W3qKolAeb0=,5,34,male,9,20110916
tKmbR4X5VXjHmxERrckawEMZ4znVy1lAQIR1vV5rdNk=,5,19,male,9,20110917
I0yFvqMoNkM8ZNHb617e1RBzIS/YRKemHO7Wj13EtA0=,13,63,male,9,20110918


In [15]:
def show_table(name, table):
    rows = table.aggregate('COUNT(*)').fetchall()[0][0]
    print(f'{name}: {rows:,} rows × {len(table.columns)} columns')
    table.limit(5).show()

show_table('train', train)
show_table('members', members)

train: 992,931 rows × 2 columns
┌──────────────────────────────────────────────┬──────────┐
│                     msno                     │ is_churn │
│                   varchar                    │  int64   │
├──────────────────────────────────────────────┼──────────┤
│ waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y= │        1 │
│ QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ= │        1 │
│ fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA= │        1 │
│ mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4= │        1 │
│ XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw= │        1 │
└──────────────────────────────────────────────┴──────────┘

members: 6,769,473 rows × 6 columns
┌──────────────────────────────────────────────┬───────┬───────┬─────────┬────────────────┬────────────────────────┐
│                     msno                     │ city  │  bd   │ gender  │ registered_via │ registration_init_time │
│                   varchar                    │ int64 │ int64 │ varchar │     int64      │      

In [16]:
con.execute('''
    CREATE OR REPLACE TEMP TABLE train_members AS
    SELECT *
    FROM train
    LEFT JOIN members USING (msno)
''')

train_members = con.table('train_members')
show_table('train_members', train_members)

train_members: 992,931 rows × 7 columns
┌──────────────────────────────────────────────┬──────────┬───────┬───────┬─────────┬────────────────┬────────────────────────┐
│                     msno                     │ is_churn │ city  │  bd   │ gender  │ registered_via │ registration_init_time │
│                   varchar                    │  int64   │ int64 │ int64 │ varchar │     int64      │         int64          │
├──────────────────────────────────────────────┼──────────┼───────┼───────┼─────────┼────────────────┼────────────────────────┤
│ e87A/+QIe/DPD/VbPbpoHHg8IApCakZM7SLsqEH44y8= │        0 │     1 │     0 │ NULL    │              7 │               20151023 │
│ qMqd+K3MzdpbKMj2vHBHo2Ugcs7jBJopCugAn+jb2cs= │        0 │     1 │     0 │ NULL    │              7 │               20151023 │
│ R2zWePnWDhAmImgqorFG4djX2Fp+DAR8iueOFy+tHjw= │        0 │     1 │     0 │ NULL    │              7 │               20151023 │
│ RMYavuIoFUtbPB11OicU/aABxftQwARGPs8eLWmsYFk= │        0 │     